# Semana 4 — Subsistema de dados geográficos

**Objetivo:** consultar municípios pela API de Localidades do IBGE sem deixar códigos de UF fixos no programa. Primeiro, consulto o endpoint de estados; depois, uso o identificador de cada UF para buscar os municípios.

API oficial: https://servicodados.ibge.gov.br/api/v1/localidades

A consulta pela internet é opcional. As Semanas 5 a 10 utilizam o SQLite entregue, e não dependem de executar este notebook novamente.


In [1]:
import requests
import pandas as pd
from IPython.display import display

API='https://servicodados.ibge.gov.br/api/v1/localidades'

def consultar_ufs(timeout=12):
    resposta=requests.get(f'{API}/estados',timeout=timeout)
    resposta.raise_for_status()
    lista=resposta.json()
    if not isinstance(lista,list):
        raise ValueError('Resposta de UFs inválida.')
    return {item['sigla'].upper():int(item['id']) for item in lista}

def consultar_municipios(sigla, mapa_ufs, timeout=12):
    if not isinstance(sigla,str) or not sigla.strip():
        raise ValueError('Informe uma sigla de UF.')
    uf=sigla.strip().upper()
    if uf not in mapa_ufs:
        raise ValueError(f'UF não encontrada: {uf}')
    resposta=requests.get(f'{API}/estados/{mapa_ufs[uf]}/municipios',timeout=timeout)
    resposta.raise_for_status()
    lista=resposta.json()
    if not isinstance(lista,list):
        raise ValueError('A resposta não é uma lista de municípios.')
    return pd.DataFrame([{'id':int(item['id']),'nome':item['nome'],'uf_sigla':uf} for item in lista])


## Execução da consulta

Coloque `CONSULTAR_API = True` ao executar com internet. Se a API estiver indisponível, a mensagem apresenta o erro; nenhum dado de exemplo é inventado.


In [2]:
CONSULTAR_API=False  # Altere para True para consultar em tempo real.

if CONSULTAR_API:
    try:
        uf_por_codigo=consultar_ufs()
        tabelas=[consultar_municipios(uf,uf_por_codigo) for uf in ('SP','RJ','MG')]
        municipios_ibr=pd.concat(tabelas,ignore_index=True)
        display(municipios_ibr.groupby('uf_sigla').size().rename('municípios'))
        display(municipios_ibr.head())
    except (requests.RequestException,ValueError,KeyError) as erro:
        print('Falha ao consultar o IBGE:',erro)
else:
    print('Consulta online desativada. Ative CONSULTAR_API para obter os municípios atuais.')


Consulta online desativada. Ative CONSULTAR_API para obter os municípios atuais.


## Validação sem consulta remota


In [3]:
for sigla in ('','XX'):
    try:
        consultar_municipios(sigla,{'SP':35})
        raise AssertionError('A validação deveria rejeitar a entrada.')
    except ValueError as erro:
        print('Entrada rejeitada:',repr(sigla),str(erro))


Entrada rejeitada: '' Informe uma sigla de UF.
Entrada rejeitada: 'XX' UF não encontrada: XX


## Conclusão

As consultas usam os códigos retornados pelo IBGE e têm timeout e tratamento de erro. Na Semana 5, o projeto passa a trabalhar com os municípios já persistidos no arquivo SQLite recebido.
